In [5]:
import sys
!{sys.executable} -m pip install optuna

In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

C:\Users\Asus\torch_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,contrast=0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])   
])

In [4]:
dataset_path = "./dataset"
dataset = datasets.ImageFolder(root=dataset_path,transform=image_transforms)

In [5]:
len(dataset)

2300

In [6]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [7]:
num_classes = len(dataset.classes)
num_classes

6

In [8]:
train_size = int(0.75 * len(dataset))
val_size = len(dataset)-train_size
train_size,val_size

(1725, 575)

In [9]:
from torch.utils.data import random_split
train_dataset, val_dataset = random_split(dataset,[train_size,val_size])

In [10]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=True)

## Model Training and Hyperparameter Training

In [11]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes,dropout_rate = 0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True

        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [14]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)

    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")

    return accuracy

In [15]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2026-03-27 14:13:10,897] A new study created in memory with name: no-name-c3357345-ab7e-4b7f-ad61-9f0ba73ad0bc
[I 2026-03-27 14:18:26,487] Trial 0 finished with value: 72.52173913043478 and parameters: {'lr': 0.003932498803937537, 'dropout_rate': 0.6118476012381513}. Best is trial 0 with value: 72.52173913043478.


Execution time: 313.7719302177429 seconds


[I 2026-03-27 14:23:31,703] Trial 1 finished with value: 73.04347826086956 and parameters: {'lr': 0.00026389242593190005, 'dropout_rate': 0.655277811359738}. Best is trial 1 with value: 73.04347826086956.


Execution time: 304.6924374103546 seconds


[I 2026-03-27 14:28:28,420] Trial 2 finished with value: 76.34782608695652 and parameters: {'lr': 0.0005621205198689565, 'dropout_rate': 0.41674820426452264}. Best is trial 2 with value: 76.34782608695652.


...

[I 2026-03-27 14:33:28,411] Trial 3 finished with value: 55.65217391304348 and parameters: {'lr': 1.5770728859331433e-05, 'dropout_rate': 0.2240898945150021}. Best is trial 2 with value: 76.34782608695652.


Execution time: 299.43075156211853 seconds


[I 2026-03-27 14:38:30,204] Trial 4 finished with value: 67.47826086956522 and parameters: {'lr': 5.206064396249684e-05, 'dropout_rate': 0.6703624853560011}. Best is trial 2 with value: 76.34782608695652.


Execution time: 301.2915093898773 seconds


[I 2026-03-27 14:40:11,627] Trial 5 pruned. 
[I 2026-03-27 14:43:31,068] Trial 6 pruned. 
[I 2026-03-27 14:45:09,875] Trial 7 pruned. 
[I 2026-03-27 14:46:48,158] Trial 8 pruned. 
[I 2026-03-27 14:48:27,895] Trial 9 pruned. 


In [16]:
study.best_params

{'lr': 0.0005621205198689565, 'dropout_rate': 0.41674820426452264}